In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import funky as f
from scipy.optimize import basinhopping
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split
from tqdm import tqdm
import matplotlib.pyplot as plt
import pytorch_lightning as pl
import corner
import emcee
rng=np.random.default_rng(seed=1)

#mu, zhd= f.sim_wrapper([1,-0.9],'prova')


n_train = 2
lp1 = 0 # low extreme param 1
lp2 = -3 # low extreme param 2
hp1 = 1 # high extreme param 1
hp2 = 0 # high extreme param 2
nome_run_nre = "NRE_00"
X = 8 ## simulations we keep


# Simulate training data
theta_samples = np.random.uniform(low=[lp1,lp2 ], high=[hp1, hp2], size=(n_train, 2))  # Parameter proposal

dataset = []
for theta in theta_samples:
    print(theta)
    mu, zhd = f.sim_wrapper(theta, nome_run_nre)
    mu = mu[:X]
    mumean, mustd = mu.mean(), mu.std()
    mu = (mu - mumean)/ mustd
    zhd = zhd[:X]
    zhdmean, zhdstd = zhd.mean(), zhd.std()
    zhd = (zhd - zhdmean)/zhdstd
    dataset.append(np.concat((mu,zhd,theta)))

dataset_tensor = torch.tensor(np.array(dataset), dtype=torch.float32)
x_samples = dataset_tensor[:, :-2]
theta_samples = dataset_tensor[:, -2:]

theta_mean = theta_samples.mean(dim=0)
theta_std = theta_samples.std(dim=0)
theta_samples = (theta_samples - theta_mean) / theta_std


def build_mlp(input_dim, hidden_dim, output_dim, layers, activation=nn.GELU()):
    """Create an MLP from the configuration."""
    seq = [nn.Linear(input_dim, hidden_dim), activation]
    for _ in range(layers):
        seq += [nn.Linear(hidden_dim, hidden_dim), activation]
    seq += [nn.Linear(hidden_dim, output_dim)]
    return nn.Sequential(*seq)


class NeuralRatioEstimator(pl.LightningModule):
    """ Simple neural likelihood-to-evidence ratio estimator, using an MLP as a parameterized classifier.
    """
    def __init__(self, x_dim, theta_dim):
        super().__init__()
        self.classifier = build_mlp(input_dim=x_dim + theta_dim, hidden_dim=128, output_dim=1, layers=4)

    def forward(self, x):
        return self.classifier(x)
    
    def loss(self, x, theta):

        # Repeat x in groups of 2 along batch axis
        x = x.repeat_interleave(2, dim=0)

        # Get a shuffled version of theta
        theta_shuffled = theta[torch.randperm(theta.shape[0])]

        # Interleave theta and shuffled theta
        theta = torch.stack([theta, theta_shuffled], dim=1).reshape(-1, theta.shape[1])

        # Get labels; ones for pairs from joint, zeros for pairs from marginals
        labels = torch.ones(x.shape[0], device=x.device) 
        labels[1::2] = 0.0

        # Pass through parameterized classifier to get logits
        logits = self(torch.cat([x, theta], dim=1))
        probs = torch.sigmoid(logits).squeeze()

        return nn.BCELoss(reduction='none')(probs, labels)


    def training_step(self, batch, batch_idx):
        x, theta = batch
        loss = self.loss(x, theta).mean()
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, theta = batch
        loss = self.loss(x, theta).mean()
        self.log("val_loss", loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=3e-4)
# Evaluate loss; initially it should be around -log(0.5) = 0.693
nre = NeuralRatioEstimator(x_dim=2*X, theta_dim=2)
nre.loss(x_samples, theta_samples)
val_fraction = 0.1
batch_size = 128
n_samples_val = int(val_fraction * len(x_samples))

dataset = TensorDataset(x_samples, theta_samples)

dataset_train, dataset_val = random_split(dataset, [len(x_samples) - n_samples_val, n_samples_val])
train_loader = DataLoader(dataset_train, batch_size=batch_size, num_workers=3, pin_memory=True, shuffle=True)
val_loader = DataLoader(dataset_val, batch_size=batch_size, num_workers=3, pin_memory=True, shuffle=False)
trainer = pl.Trainer(max_epochs=20)
trainer.fit(model=nre, train_dataloaders=train_loader, val_dataloaders=val_loader);

def log_like(theta, x):
    """ Log-likelihood ratio estimator using trained classifier logits. """
    x = torch.Tensor(x)
    theta = torch.Tensor(theta)

    x = (x - torch.mean(x)) / torch.std(x)
    theta = (theta - torch.mean(theta)) / torch.std(theta)

    x = torch.atleast_1d(x)
    theta = torch.atleast_1d(theta)

    with torch.no_grad():
        return nre.classifier(torch.cat([x, theta], dim=-1)).squeeze().item()



theta_test = np.array([0.34, -1])

mu_test, zhd_test = f.sim_wrapper(theta_test, nome_run_nre)

mu_test = mu_test[:X]
mu_test = (mu_test - mu_test.mean()) / mu_test.std()

zhd_test = zhd_test[:X]
zhd_test = (zhd_test - zhd_test.mean()) / zhd_test.std()

x_test = np.concatenate((mu_test, zhd_test))

log_like(theta_test, x_test)

def log_post(theta, x):
    """ Log-posterior distribution, for sampling.
    """
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    else:
        return lp + log_like(theta, x)

def log_prior(theta):
    if (0 <= theta[0] <= 1) and (-3 <= theta[1] <= 0):
        return 0.0
    return -np.inf

initial_guess = [0.5, -1.5]
minimizer_kwargs = {"method": "L-BFGS-B", "bounds": ((0, 1), (-3, 0))}

opt = basinhopping(
    lambda thetas: -float(log_like(thetas, x_test)), 
    initial_guess, 
    minimizer_kwargs=minimizer_kwargs
)
print(f"MLE parameters: {opt.x}; true parameters: {theta_test}")
ndim, nwalkers = 2, 32

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_post, args=(x_test,))

pos = opt.x + 1e-3 * np.random.randn(nwalkers, ndim)
sampler.run_mcmc(pos, 5000, progress=True)

flat_samples = sampler.get_chain(discard=1000, flat=True)

corner.corner(
    flat_samples, 
    labels=["param_1", "param_2"], 
    truths=theta_test, 
    smooth=1.
);